In [1]:
!pip install pyspark -q
from google.colab import drive
drive.mount('/content/drive', force_remount=True)

import os, datetime
from pyspark.sql import SparkSession
from pyspark.sql import functions as F
from pyspark.sql.window import Window
from pyspark.ml.feature import Word2Vec

# 1. Cấu hình đường dẫn
BASE_PATH = "/content/drive/MyDrive/HM-DATA/"
INPUT_FILE = BASE_PATH + "processed/cleaned_transactions.parquet"
ARTICLES_FILE = BASE_PATH + "processed/articles_processed.parquet"
CUSTOMERS_FILE = BASE_PATH + "processed/customers_processed.parquet"
OUTPUT_DIR = BASE_PATH + "outputs/candidates/"
os.makedirs(OUTPUT_DIR, exist_ok=True)

# 2. Khởi tạo Spark tối ưu RAM
spark = SparkSession.builder \
    .appName("HM_Metadata_Optimized_Recall") \
    .config("spark.driver.memory", "10g") \
    .getOrCreate()

# 3. Đọc dữ liệu
transactions = spark.read.parquet(INPUT_FILE)
articles = spark.read.parquet(ARTICLES_FILE).withColumn("article_id", F.lpad(F.col("article_id").cast("string"), 10, "0"))
customers = spark.read.parquet(CUSTOMERS_FILE)

# 4. Tính toán mốc thời gian Tuần 7 (Validation)
max_date = transactions.select(F.max("t_dat")).collect()[0][0]
test_start_date = max_date - datetime.timedelta(days=7)
val_start_date = test_start_date - datetime.timedelta(days=7)

# Lấy 6 tuần đầu để làm Profile
train_data = transactions.filter(F.col("t_dat") < F.lit(val_start_date)) \
    .withColumn("article_id", F.lpad(F.col("article_id").cast("string"), 10, "0"))

print(f"✅ Spark Ready! Đang xử lý cho {customers.count():,} khách hàng.")

Mounted at /content/drive
✅ Spark Ready! Đang xử lý cho 1,371,980 khách hàng.


In [2]:
# 1. Tìm Category hot nhất sàn (Cứu cánh Cold-start)
top_global_cat = articles.join(train_data, "article_id") \
    .groupBy("product_group_name").count() \
    .orderBy(F.desc("count")).limit(1).collect()[0][0]

# 2. Tìm Category yêu thích của khách cũ
user_fav_cat = train_data.join(F.broadcast(articles.select("article_id", "product_group_name")), "article_id") \
    .groupBy("customer_id", "product_group_name").count()

user_profile = user_fav_cat.withColumn("rn", F.row_number().over(Window.partitionBy("customer_id").orderBy(F.desc("count")))) \
    .filter(F.col("rn") == 1).select("customer_id", "product_group_name")

# 3. Gán Category mặc định cho khách hàng mới (Cold-start)
user_profile_full = customers.select("customer_id").join(user_profile, "customer_id", "left") \
    .fillna({"product_group_name": top_global_cat})

# 4. Top 15 món Trending của mỗi Category (Tuần 6)
w6_start = val_start_date - datetime.timedelta(days=7)
top_trending_by_cat = train_data.filter(F.col("t_dat") >= F.lit(w6_start)) \
    .join(F.broadcast(articles.select("article_id", "product_group_name")), "article_id") \
    .groupBy("product_group_name", "article_id").count() \
    .withColumn("rn", F.row_number().over(Window.partitionBy("product_group_name").orderBy(F.desc("count")))) \
    .filter(F.col("rn") <= 15).select("product_group_name", "article_id")

# Ứng viên Agg & Seasonal (Thiết lập Priority 1)
cand_agg_seasonal = user_profile_full.join(F.broadcast(top_trending_by_cat), "product_group_name") \
    .select("customer_id", "article_id", F.lit(1).alias("priority"))

print(f"✅ Đã tạo xong ứng viên Agg & Seasonal.")

✅ Đã tạo xong ứng viên Agg & Seasonal.


In [3]:
# --- 1. Huấn luyện Word2Vec siêu nhẹ (Lên làm Priority 0) ---
sequences = train_data.groupBy("customer_id").agg(F.collect_list("article_id").alias("item_list")).filter(F.size("item_list") >= 2)
w2v = Word2Vec(vectorSize=16, minCount=2, inputCol="item_list", outputCol="model_vector")
w2v_model = w2v.fit(sequences)
item_vectors = w2v_model.getVectors()

# Lấy món cuối cùng của khách để tìm similarity
window_last = Window.partitionBy("customer_id").orderBy(F.desc("t_dat"))
last_items = train_data.withColumn("rn", F.row_number().over(window_last)) \
    .filter(F.col("rn") == 1).select("customer_id", "article_id")

cand_similarity = last_items.join(item_vectors, last_items.article_id == item_vectors.word) \
    .select("customer_id", "article_id", F.lit(0).alias("priority"))

# --- 2. Same Model - Cùng mẫu khác màu (Priority 2) ---
last_items_7 = last_items.withColumn("p_code", F.substring(F.col("article_id"), 1, 7))
all_variants = articles.select(F.col("article_id").alias("v_id"), F.substring(F.col("article_id"), 1, 7).alias("p_code"))

cand_same_model = last_items_7.join(F.broadcast(all_variants), "p_code") \
    .filter(F.col("article_id") != F.col("v_id")) \
    .select("customer_id", F.col("v_id").alias("article_id"), F.lit(2).alias("priority"))

print("✅ Đã tạo xong ứng viên Similarity (P0) và Same Model (P2).")

✅ Đã tạo xong ứng viên Similarity (P0) và Same Model (P2).


In [4]:
# 1. Gộp 3 nguồn
all_meta = cand_similarity.union(cand_agg_seasonal).union(cand_same_model)

# 2. Xếp hạng và phân bổ theo phễu ưu tiên
window_limit = Window.partitionBy("customer_id", "priority").orderBy(F.lit(1))
all_meta_refined = all_meta.withColumn("internal_rn", F.row_number().over(window_limit)) \
    .filter(
        ((F.col("priority") == 0) & (F.col("internal_rn") <= 12)) | # Word2Vec lấy 12
        ((F.col("priority") == 1) & (F.col("internal_rn") <= 10)) | # Agg/Seasonal lấy 10
        ((F.col("priority") == 2) & (F.col("internal_rn") <= 8))    # Same Model lấy 8
    )

# 3. Lấy Top 30 duy nhất cho mỗi khách
final_window = Window.partitionBy("customer_id").orderBy("priority", "internal_rn")
meta_candidates_final = all_meta_refined.dropDuplicates(["customer_id", "article_id"]) \
    .withColumn("rank", F.row_number().over(final_window)) \
    .filter(F.col("rank") <= 30) \
    .groupBy("customer_id").agg(F.collect_list("article_id").alias("meta_candidates"))

# 4. Lưu file cuối cùng
meta_candidates_final.write.mode("overwrite").parquet(OUTPUT_DIR + "meta_candidates_pro_W7.parquet")

print(f"🏆 XUẤT FILE THÀNH CÔNG: meta_candidates_pro_W7.parquet")
print(f"📊 Tổng số khách hàng bao phủ: {meta_candidates_final.count():,}")

🏆 XUẤT FILE THÀNH CÔNG: meta_candidates_pro_W7.parquet
📊 Tổng số khách hàng bao phủ: 1,371,980


In [5]:
# 1. Ground Truth (Tuần 7)
ground_truth = transactions.filter((F.col("t_dat") >= F.lit(val_start_date)) & (F.col("t_dat") < F.lit(test_start_date))) \
    .select("customer_id", F.lpad(F.col("article_id").cast("string"), 10, "0").alias("article_id"))

actual_counts = ground_truth.groupBy("customer_id").count().withColumnRenamed("count", "actual_cnt")

print(f"✅ Đã chuẩn bị xong Ground Truth cho {actual_counts.count():,} khách hàng.")

✅ Đã chuẩn bị xong Ground Truth cho 74,575 khách hàng.


In [6]:
# 1. Hàm tính Recall
def evaluate_source(source_df, limit=10):
    window_spec = Window.partitionBy("customer_id").orderBy("priority")
    cand_exploded = source_df.withColumn("rn", F.row_number().over(window_spec)) \
                             .filter(F.col("rn") <= limit).select("customer_id", "article_id")

    hits = ground_truth.join(cand_exploded, ["customer_id", "article_id"], "inner") \
                       .groupBy("customer_id").count().withColumnRenamed("count", "hit_cnt")

    recall_df = actual_counts.join(hits, "customer_id", "left").fillna(0)
    return recall_df.select(F.avg(F.col("hit_cnt") / F.col("actual_cnt"))).collect()[0][0]

# 2. Phân tích
print("🚀 Đang phân tích hiệu quả Metadata (Bản tối ưu Word2Vec)...")
recall_total = evaluate_source(all_meta.dropDuplicates(["customer_id", "article_id"]), limit=30)
recall_w2v = evaluate_source(cand_similarity, limit=10)
recall_agg = evaluate_source(cand_agg_seasonal, limit=10)
recall_same = evaluate_source(cand_same_model, limit=10)

print("\n" + "="*55)
print(f"📊 BÁO CÁO HIỆU SUẤT METADATA PRO (Word2Vec Priority 0)")
print("-" * 55)
print(f"{'1. Word2Vec Similarity (P0)':<35} | {recall_w2v:.6f}")
print(f"{'2. Aggregation & Seasonal (P1)':<35} | {recall_agg:.6f}")
print(f"{'3. Same Model / 7-digit (P2)':<35} | {recall_same:.6f}")
print("-" * 55)
print(f"{'🔥 TỔNG CỘNG RECALL@30':<35} | {recall_total:.6f}")
print("="*55)

🚀 Đang phân tích hiệu quả Metadata (Bản tối ưu Word2Vec)...

📊 BÁO CÁO HIỆU SUẤT METADATA PRO (Word2Vec Priority 0)
-------------------------------------------------------
1. Word2Vec Similarity (P0)         | 0.013396
2. Aggregation & Seasonal (P1)      | 0.012660
3. Same Model / 7-digit (P2)        | 0.009449
-------------------------------------------------------
🔥 TỔNG CỘNG RECALL@30               | 0.045971
